In [277]:
import glob
import logging
import os
import warnings
from collections.abc import Callable, Iterator
from functools import partial
from pathlib import Path
from typing import Any, Union

import dask
import dask.array as da
import numpy as np
import pandas as pd
import pkg_resources
from dask.array.core import normalize_chunks
from scipy.ndimage import affine_transform
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import LineString
from shapely.geometry.base import BaseGeometry
from shapely.geometry.polygon import Polygon
from shapely.strtree import STRtree
from skimage.io import imread, imsave
from skimage.transform import AffineTransform
from tqdm.auto import tqdm

from macrohet.dataio import read_harmony_metadata

# ignore shapely depreciation warning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)
pkg_resources.require("Shapely<2.0.0")
# ignore error message for pandas new col assignment
pd.options.mode.chained_assignment = None
FilePath = Path | str
ArrayLike = Union[np.ndarray, "dask.array.Array"]

logging.basicConfig(level=logging.INFO)


class FileNotFoundError(Exception):

    pass


def find_files_exist(fns: list[str], image_dir: str):

    for fn in fns:
        file_path = os.path.join(image_dir, fn)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"The file '{file_path}' does not exist.")


def compile_mosaic(
    image_dir: os.PathLike,
    metadata: pd.DataFrame,
    row: int,
    col: int,
    input_transforms: list[Callable[[np.ndarray], np.ndarray]] | None = None,
    set_plane: Any | None = None,  # Can be int or 'max_proj'/'sum_proj'
    set_channel: int | None = None,
    set_time: int | None = None,
    overlap_percentage: float = 0.1,
    subset_field_IDs: list[int] | None = None,
    n_tile_rows: int | None = None,
    n_tile_cols: int | None = None
) -> np.ndarray:


    # check if specified row and column exists by checking metadata
    if str(row) not in metadata['Row'].unique():
        raise ValueError("Row not found in metadata.")
    if str(col) not in metadata['Col'].unique():
        raise ValueError("Column not found in metadata.")

    # check if projection is to be conducted over Z
    if isinstance(set_plane, str):
        # if set_plane is str, then specify which type of projection to
        # to conduct, also check that input type is accepted
        if set_plane not in ['max_proj', 'sum_proj']:
            raise TypeError("""Please specify either 'max_proj' or 'sum_proj'
                            if you want a projection over Z axis,
                            else specify 'set_plane' as an integer""")
        projection = set_plane
        set_plane = None
    else:
        projection = None
    # extract some necessary information from the metadata before tiling
    channel_IDs = (metadata['ChannelID'].unique()
                   if set_channel is None else [set_channel])
    plane_IDs = (metadata['PlaneID'].unique()
                 if set_plane is None else [set_plane])
    timepoint_IDs = (metadata['TimepointID'].unique()
                     if set_time is None else [set_time])

    # take a sample image to find dtype
    sample_fn = metadata['URL'][(metadata['Row'] == str(row))
                                & (metadata['Col'] == str(col))].iloc[0]
    dtype = imread(image_dir + f'/{sample_fn}').dtype

    # use metadata and overlap percentage to calculate the final expected size
    if subset_field_IDs:
        number_tiles = len(subset_field_IDs)
    else:
        number_tiles = int(metadata['FieldID'].max())

    # if n_tile_rows or cols are not supplied, then assumme mosaic is square
    if not n_tile_rows:
        n_tile_rows = n_tile_cols = np.sqrt(number_tiles)

    tile_size = int(metadata['ImageSizeX'].max())
    image_size = final_image_size(tile_size, overlap_percentage,
                                  n_tile_rows, n_tile_cols)

    load_transform_image = partial(load_image, transforms=input_transforms)

    # stitch the images together over all defined axis using dask delayed
    images = [dask.delayed(stitch)(load_transform_image,
                                   metadata,
                                   image_dir,
                                   time,
                                   plane,
                                   channel,
                                   str(row),
                                   str(col),
                                   n_tile_rows,
                                   n_tile_cols,
                                   subset_field_IDs)[0]
              for time in timepoint_IDs
              for channel in channel_IDs
              for plane in plane_IDs]

    # create a series of dask arrays out of the delayed funcs
    images = [da.from_delayed(frame,
                              shape=image_size,
                              dtype=dtype)
              # for frame in tqdm(images, desc='Stitching images together')]
              for frame in images]

    # rechunk so they are more managable along original image tile size
    images = [frame.rechunk(tile_size, tile_size) for frame in images]
    # stack them together and call compute so the it returns a single da and not a da of a da
    images = da.stack(images, axis=0)  # .compute()
    
    print("Expected reshape:", (len(timepoint_IDs), len(channel_IDs), len(plane_IDs), images.shape[-2], images.shape[-1]))
    # reshape them according to TCZXY
    images = images.reshape((len(timepoint_IDs),
                             len(channel_IDs),
                             len(plane_IDs),
                             images.shape[-2], images.shape[-1]))
    # conduct projection according to specified type
    if projection == 'max_proj':
        images = np.max(images, axis=2)
    # sum projection requires image clipping in case px value exceeds dtype max
    elif projection == 'sum_proj':
        # Perform the summed projection along the z-axis
        summed_projection = np.sum(images, axis=2)

        # Determine the maximum value based on the data type
        max_value = np.iinfo(dtype).max

        # Clip the pixel values that exceed the maximum representable value
        images = np.clip(summed_projection, 0, max_value).astype(dtype)

    return images


def stitch(load_transform_image: partial,
           df: pd.DataFrame,
           image_dir: str,
           time: int,
           plane: int,
           channel: int,
           row: int,
           col: int,
           n_tile_rows: int,
           n_tile_cols: int,
           image_size: tuple,
           subset_field_IDs=None,) -> tuple[da.Array, list[tuple]]:

    # Filter metadata for the current mosaic
    conditions = (df['TimepointID'] == str(time)) & (df['PlaneID'] == str(plane)) & \
                 (df['ChannelID'] == str(channel)) & (df['Row'] == str(row)) & (df['Col'] == str(col))
    filtered_df = df[conditions]

    if subset_field_IDs:
        filtered_df = filtered_df[filtered_df['FieldID'].isin(subset_field_IDs)]

    # Extract filenames
    fns = filtered_df['URL']

    # Check if files exist
    find_files_exist(fns, image_dir)
    fns = [glob.glob(os.path.join(image_dir, fn))[0] for fn in fns]

    # Load and transform images
    sample = imread(fns[0])

    # Define function to fuse the image
    _fuse_func = partial(fuse_func, imload_fn=load_transform_image, dtype=sample.dtype)

    # Convert coordinates from standard units to pixels
    coords = filtered_df[["URL", "PositionX", "PositionY", "ImageResolutionX", "ImageResolutionY"]]
    coords['PositionXPix'] = coords['PositionX'].astype(float) / coords['ImageResolutionX'].astype(float)
    coords['PositionYPix'] = coords['PositionY'].astype(float) / coords['ImageResolutionY'].astype(float)
    # Shift origin so top-left tile is at (0, 0)
    # coords['PositionXPix'] -= coords['PositionXPix'].min()
    # coords['PositionYPix'] -= coords['PositionYPix'].min()

    norm_coords = list(zip(coords['PositionXPix'], coords['PositionYPix']))

    # Convert tile coordinates to transformation matrices and shift to the origin
    transforms = [AffineTransform(translation=stage_coord).params for stage_coord in norm_coords]
    tiles = [transform_tile_coord(sample.shape, transform) for transform in transforms]
    all_bboxes = np.vstack(tiles)
    stitched_shape =  tuple(np.round(all_bboxes.max(axis=0) - all_bboxes.min(axis=0)).astype(int))

    shift_to_origin = AffineTransform(translation=-all_bboxes.min(axis=0))
    transforms_with_shift = [t @ shift_to_origin.params for t in transforms]
    shifted_tiles = [transform_tile_coord(sample.shape, t) for t in transforms_with_shift]

    # Determine chunk size and boundaries
    chunk_size = (int(stitched_shape[0] / n_tile_rows), int(stitched_shape[1] / n_tile_cols))
    chunks = normalize_chunks(chunk_size, shape=stitched_shape)
    assert np.all(np.array(stitched_shape) == np.array(list(map(sum, chunks)))), "Chunks do not fit into mosaic size"
    chunk_boundaries = list(get_chunk_coord(stitched_shape, chunk_size))

    # Use Shapely to find the intersection of the chunks
    tiles_shifted_shapely = [numpy_shape_to_shapely(s) for s in shifted_tiles]
    chunk_shapes = [get_rect_from_chunk_boundary(b) for b in chunk_boundaries]
    chunks_shapely = [numpy_shape_to_shapely(c) for c in chunk_shapes]

    # Build dictionary of chunk shape data with filenames and transformations
    for tile_shifted_shapely, file, transform in zip(tiles_shifted_shapely, fns, transforms_with_shift):
        tile_shifted_shapely.fuse_info = {'file': file, 'transform': transform}
    for chunk_shapely, chunk_boundary in zip(chunks_shapely, chunk_boundaries):
        chunk_shapely.fuse_info = {'chunk_boundary': chunk_boundary}

    chunk_tiles = find_chunk_tile_intersections(tiles_shifted_shapely, chunks_shapely)

    # Tile images together
    frame = da.map_blocks(func=_fuse_func, chunks=chunks, input_tile_info=chunk_tiles, dtype=sample.dtype)
    frame = da.rot90(frame)  # Need this to bridge cartesian coords with python image coords?
    
    return frame, tiles_shifted_shapely


def transform_tile_coord(shape: tuple[int, int], affine_matrix: np.ndarray) -> np.ndarray:

    h, w = shape
    baserect = np.array([[0, 0], [h, 0], [h, w], [0, w]])
    augmented_baserect = np.concatenate((baserect, np.ones((baserect.shape[0], 1))), axis=1)
    transformed_rect = (affine_matrix @ augmented_baserect.T).T[:, :-1]
    return transformed_rect


def get_chunk_coord(shape: tuple[int, int], chunk_size: tuple[int, int]) -> Iterator[tuple[tuple[int, int], tuple[int, int]]]:

    chunksy, chunksx = normalize_chunks(chunk_size, shape=shape)
    y = 0
    for cy in chunksy:
        x = 0
        for cx in chunksx:
            yield ((y, y + cy), (x, x + cx))
            x += cx
        y += cy


def numpy_shape_to_shapely(coords: np.ndarray, shape_type: str = "polygon") -> BaseGeometry:

    _coords = coords[:, ::-1].copy()
    _coords[:, 1] *= -1
    if shape_type in ("rectangle", "polygon", "ellipse"):
        return Polygon(_coords)
    elif shape_type in ("line", "path"):
        return LineString(_coords)
    else:
        raise ValueError("Invalid shape type")


def get_rect_from_chunk_boundary(chunk_boundary: tuple[tuple[int, int], tuple[int, int]]) -> np.ndarray:
    """Given a chunk boundary tuple, return a numpy array representing a rectangle.

    Parameters
    ----------
    chunk_boundary : Tuple[Tuple[int, int], Tuple[int, int]]
        Chunk boundary.

    Returns
    -------
    np.ndarray
        Rectangle coordinates.

    """
    ylim, xlim = chunk_boundary
    miny, maxy = ylim[0], ylim[1] - 1
    minx, maxx = xlim[0], xlim[1] - 1
    return np.array([[miny, minx], [maxy, minx], [maxy, maxx], [miny, maxx]])


def find_chunk_tile_intersections(
    tiles_shapely: list[BaseGeometry],
    chunks_shapely: list[BaseGeometry]
) -> dict[tuple[int, int], list[tuple[str | np.ndarray, np.ndarray]]]:
    """For each output array chunk, find the intersecting image tiles.

    Parameters
    ----------
    tiles_shapely : List[BaseGeometry]
        List of shapely objects corresponding to image tiles.
    chunks_shapely : List[BaseGeometry]
        List of shapely objects representing dask array chunks.

    Returns
    -------
    Dict[Tuple[int, int], List[Tuple[Union[str, np.ndarray], np.ndarray]]]
        Dictionary mapping chunk anchor points to tuples of image file names and their corresponding affine transform matrices.

    """
    chunk_to_tiles = {}
    tile_tree = STRtree(tiles_shapely)

    for chunk_shape in chunks_shapely:
        chunk_boundary = chunk_shape.fuse_info["chunk_boundary"]
        anchor_point = (chunk_boundary[0][0], chunk_boundary[1][0])
        intersecting_tiles = tile_tree.query(chunk_shape)
        chunk_to_tiles[anchor_point] = [
            ((t.fuse_info["file"], t.fuse_info["transform"]))
            for t in intersecting_tiles
        ]
    return chunk_to_tiles


def fuse_func(
    input_tile_info: dict[tuple[int, int], list[tuple[str | Path | np.ndarray, np.ndarray]]],
    imload_fn: Callable | None = imread,
    block_info=None,
    dtype=np.uint16,
) -> np.ndarray:
    """Fuses the tiles that intersect the current chunk of a dask array using maximum projection.

    Parameters
    ----------
    input_tile_info : Dict[Tuple[int, int], List[Tuple[Union[str, Path, np.ndarray], np.ndarray]]]
        Information about the input tiles.
    imload_fn : Optional[Callable], optional
        Function to load the images, by default imread.
    block_info : optional
        Information about the dask block, by default None.
    dtype : data-type, optional
        The desired data-type for the array, by default np.uint16.

    Returns
    -------
    np.ndarray
        Array of chunk-shape containing max projection of tiles falling into chunk.

    """
    array_location = block_info[None]["array-location"]
    anchor_point = (array_location[0][0], array_location[1][0])
    chunk_shape = block_info[None]["chunk-shape"]
    tiles_info = input_tile_info[anchor_point]
    fused = np.zeros(chunk_shape, dtype=dtype)

    for image_representation, tile_affine in tiles_info:
        if imload_fn is not None:
            tile_path = image_representation
            im = imload_fn(tile_path)
        else:
            im = image_representation

        shift = AffineTransform(translation=(-anchor_point[0], -anchor_point[1]))
        tile_shifted = affine_transform(
            im,
            matrix=np.linalg.inv(shift.params @ tile_affine),
            output_shape=chunk_shape,
            cval=0,
        )
        fused = np.maximum(fused, tile_shifted.astype(dtype))

    return fused


def load_image(file: str | Path, transforms: list[Callable[[np.ndarray], np.ndarray]] = None) -> np.ndarray:
    """Load image from given filepath with optional transformation implementation.

    Parameters
    ----------
    file : Union[str, Path]
        Path to the image file.
    transforms : List[Callable[[np.ndarray], np.ndarray]], optional
        List of transformation functions to apply to the image, by default None.

    Returns
    -------
    np.ndarray
        Loaded and possibly transformed image.

    """
    try:
        img = imread(file)
    except Exception as e:
        raise Exception(f'{e} \n Could not load file: {file}') from e

    img = da.rot90(img, k=3)  # Need this to bridge cartesian coords with python image coords

    if transforms is not None:
        for transform in transforms:
            img = transform(img)

    return img


def final_image_size(size_of_tile, overlap_percentage, n_tile_rows, n_tile_cols):
    """Calculate the size of the final stitched image for a rectangular mosaic.

    Parameters
    ----------
    n_tile_rows (int): Number of tiles along the width.
    n_tile_cols (int): Number of tiles along the height.
    size_of_tile (int): Size of each tile in pixels.
    overlap_percentage (float): Overlap between the tiles as a percentage.

    Returns
    -------
    tuple: Size of the final stitched image in pixels (width, height).

    """
    # Calculate the actual overlap in pixels
    overlap = overlap_percentage * size_of_tile
    
    # Calculate the size of the final stitched image in width
    final_image_width = (n_tile_cols * size_of_tile) - ((n_tile_cols - 1) * overlap)

    # Calculate the size of the final stitched image in height
    final_image_height = (n_tile_rows * size_of_tile) - ((n_tile_rows - 1) * overlap)

    return (int(final_image_width), int(final_image_height))


In [278]:
metadata = read_harmony_metadata('/home/dayn/analysis/macrohet/data/untiled_images/Index_0305.idx.xml')

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [251]:
metadata['PositionX'].unique()

array(['0', '-0.000290616', '0.000290616'], dtype=object)

In [279]:
import numpy as np
import pandas as pd

# These are in pixels, so scale down for downsampled image
pixel_fields = ['ImageSizeX', 'ImageSizeY']
metadata[pixel_fields] = (
    metadata[pixel_fields]
    .apply(pd.to_numeric, errors='coerce')
    .div(5.04)
    .apply(np.floor)
    .astype(int)
    .astype(str)
)

# These are in microns, and used in Position / Resolution → scale Resolution *up*
resolution_fields = ['ImageResolutionX', 'ImageResolutionY']
metadata[resolution_fields] = (
    metadata[resolution_fields]
    .apply(pd.to_numeric, errors='coerce')
    .mul(5.04)
    .round(10)
    .astype(str)
)

# DO NOT scale PositionX / PositionY — they're in microns and must stay precise


In [272]:
image_dir = 'data/untiled_images/'


In [273]:
row=3
col=5
n_tile_cols=3
n_tile_rows=3
# Optional arguments
input_transforms = None
set_plane = None # could also be int like 0
set_channel = None
set_time = None
overlap_percentage = 0.1
subset_field_IDs = None  # or e.g. [1,2,3]

In [280]:

# extract some necessary information from the metadata before tiling
channel_IDs = (metadata['ChannelID'].unique()
               if set_channel is None else [set_channel])
plane_IDs = (metadata['PlaneID'].unique()
             if set_plane is None else [set_plane])
timepoint_IDs = (metadata['TimepointID'].unique()
                 if set_time is None else [set_time])

# take a sample image to find dtype
sample_fn = metadata['URL'][(metadata['Row'] == str(row))
                            & (metadata['Col'] == str(col))].iloc[0]
dtype = imread(image_dir + f'/{sample_fn}').dtype

tile_size = int(metadata['ImageSizeX'].max())
image_size = final_image_size(tile_size, overlap_percentage,
                              n_tile_rows, n_tile_cols)
print(f"[DEBUG] expected image size: {image_size}")

load_transform_image = partial(load_image, transforms=input_transforms)

# stitch the images together over all defined axis using dask delayed
images = [dask.delayed(stitch)(load_transform_image,
                               metadata,
                               image_dir,
                               time,
                               plane,
                               channel,
                               str(row),
                               str(col),
                               n_tile_rows,
                               n_tile_cols,
                               image_size,
                               subset_field_IDs)[0]
          for time in timepoint_IDs
          for channel in channel_IDs
          for plane in plane_IDs]

# create a series of dask arrays out of the delayed funcs
images = [da.from_delayed(frame,
                          shape=image_size,
                          dtype=dtype)
          # for frame in tqdm(images, desc='Stitching images together')]
          for frame in images]

# rechunk so they are more managable along original image tile size
images = [frame.rechunk(tile_size, tile_size) for frame in images]
# stack them together and call compute so the it returns a single da and not a da of a da
images = da.stack(images, axis=0)  # .compute()

print("Expected reshape:", (len(timepoint_IDs), len(channel_IDs), len(plane_IDs), images.shape[-2], images.shape[-1]))
# reshape them according to TCZXY
images = images.reshape((len(timepoint_IDs),
                         len(channel_IDs),
                         len(plane_IDs),
                         images.shape[-2], images.shape[-1]))


[DEBUG] expected image size: (1198, 1198)
Expected reshape: (75, 2, 3, 1198, 1198)


In [281]:
frame = images[0,0,0].compute().compute()

In [283]:
viewer.add_image(frame)

<Image layer 'frame [2]' at 0x7fda2edb89d0>

In [257]:
frame

array([[  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       ...,
       [575, 402, 413, ...,   0,   0,   0],
       [497, 333, 358, ...,   0,   0,   0],
       [454, 293, 243, ...,   0,   0,   0]],
      shape=(1198, 1198), dtype=uint16)

In [258]:
# frame is your 2D numpy array of shape (1198, 1198)
nonzero_y, nonzero_x = np.nonzero(frame)

# Compute bounds
min_y, max_y = nonzero_y.min(), nonzero_y.max()
min_x, max_x = nonzero_x.min(), nonzero_x.max()

# Compute dimensions
height = max_y - min_y + 1
width = max_x - min_x + 1

print(f"Non-zero bounding box:")
print(f"  X: {min_x} → {max_x} ({width} px)")
print(f"  Y: {min_y} → {max_y} ({height} px)")

Non-zero bounding box:
  X: 0 → 1008 (1009 px)
  Y: 189 → 1197 (1009 px)


In [266]:
%%time
loaded_images = images.compute().compute()

CPU times: user 17min 23s, sys: 39.5 s, total: 18min 2s
Wall time: 7min 43s


In [64]:
import napari


In [268]:
# viewer = napari.Viewer()
viewer.add_image(loaded_images, channel_axis=1)

[<Image layer 'Image' at 0x7fda16de37f0>,
 <Image layer 'Image [1]' at 0x7fda16de23e0>]

Error calling Python override of QOpenGLWidget::event(): Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 655, in event
    out = super(QtBaseCanvasBackend, self).event(ev)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 655, in event
    out = super(QtBaseCanvasBackend, self).event(ev)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 963, in paintGL
    self._vispy_canvas.events.draw(region=None)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/sit

In [217]:
viewer.layers['Points'].data

array([[ 56.        ,   0.        ,   1.        , 479.43254366,
        290.29378351],
       [ 56.        ,   0.        ,   1.        , 615.25530021,
        427.54728442]])

In [218]:
viewer.layers['Points'].data[1]-viewer.layers['Points'].data[0]

array([  0.        ,   0.        ,   0.        , 135.82275656,
       137.25350091])

# Debuggin stitch

In [196]:
time = 1
plane = 1
channel = 1
df = metadata

In [221]:
# Filter metadata for the current mosaic
conditions = (df['TimepointID'] == str(time)) & (df['PlaneID'] == str(plane)) & \
             (df['ChannelID'] == str(channel)) & (df['Row'] == str(row)) & (df['Col'] == str(col))
filtered_df = df[conditions]

if subset_field_IDs:
    filtered_df = filtered_df[filtered_df['FieldID'].isin(subset_field_IDs)]

# Extract filenames
fns = filtered_df['URL']

# Check if files exist
find_files_exist(fns, image_dir)
fns = [glob.glob(os.path.join(image_dir, fn))[0] for fn in fns]

# Load and transform images
sample = imread(fns[0])

# Define function to fuse the image
_fuse_func = partial(fuse_func, imload_fn=load_transform_image, dtype=sample.dtype)

# Convert coordinates from standard units to pixels
coords = filtered_df[["URL", "PositionX", "PositionY", "ImageResolutionX", "ImageResolutionY"]]
coords['PositionXPix'] = coords['PositionX'].astype(float) / coords['ImageResolutionX'].astype(float)
coords['PositionYPix'] = coords['PositionY'].astype(float) / coords['ImageResolutionY'].astype(float)
# Shift origin so top-left tile is at (0, 0)
coords['PositionXPix'] -= coords['PositionXPix'].min()
coords['PositionYPix'] -= coords['PositionYPix'].min()
norm_coords = list(zip(coords['PositionXPix'], coords['PositionYPix']))
print("\n[DEBUG] Raw positions and resolutions:")
print(coords[['PositionX', 'ImageResolutionX', 'PositionXPix']].head(9))

# Convert tile coordinates to transformation matrices and shift to the origin
transforms = [AffineTransform(translation=stage_coord).params for stage_coord in norm_coords]
tiles = [transform_tile_coord(sample.shape, transform) for transform in transforms]
all_bboxes = np.vstack(tiles)
stitched_shape = (1198,1198)#tuple(np.round(all_bboxes.max(axis=0) - all_bboxes.min(axis=0)).astype(int))

shift_to_origin = AffineTransform(translation=-all_bboxes.min(axis=0))
transforms_with_shift = [t @ shift_to_origin.params for t in transforms]
shifted_tiles = [transform_tile_coord(sample.shape, t) for t in transforms_with_shift]

# Determine chunk size and boundaries
chunk_size = (int(stitched_shape[0] / n_tile_rows), int(stitched_shape[1] / n_tile_cols))
chunks = normalize_chunks(chunk_size, shape=stitched_shape)
assert np.all(np.array(stitched_shape) == np.array(list(map(sum, chunks)))), "Chunks do not fit into mosaic size"
chunk_boundaries = list(get_chunk_coord(stitched_shape, chunk_size))

# Use Shapely to find the intersection of the chunks
tiles_shifted_shapely = [numpy_shape_to_shapely(s) for s in shifted_tiles]
chunk_shapes = [get_rect_from_chunk_boundary(b) for b in chunk_boundaries]
chunks_shapely = [numpy_shape_to_shapely(c) for c in chunk_shapes]

# Build dictionary of chunk shape data with filenames and transformations
for tile_shifted_shapely, file, transform in zip(tiles_shifted_shapely, fns, transforms_with_shift):
    tile_shifted_shapely.fuse_info = {'file': file, 'transform': transform}
for chunk_shapely, chunk_boundary in zip(chunks_shapely, chunk_boundaries):
    chunk_shapely.fuse_info = {'chunk_boundary': chunk_boundary}

chunk_tiles = find_chunk_tile_intersections(tiles_shifted_shapely, chunks_shapely)

# Tile images together
frame = da.map_blocks(func=_fuse_func, chunks=chunks, input_tile_info=chunk_tiles, dtype=sample.dtype)
frame = da.rot90(frame)  # Need this to bridge cartesian coords with python image coords?



[DEBUG] Raw positions and resolutions:
        PositionX ImageResolutionX  PositionXPix
54              0            1e-06       290.616
60   -0.000290616            1e-06         0.000
66              0            1e-06       290.616
72    0.000290616            1e-06       581.232
78    0.000290616            1e-06       581.232
84              0            1e-06       290.616
90   -0.000290616            1e-06         0.000
96   -0.000290616            1e-06         0.000
102   0.000290616            1e-06       581.232


In [223]:
581.232+428

1009.232

In [224]:
coords

,URL,PositionX,PositionY,ImageResolutionX,ImageResolutionY,PositionXPix,PositionYPix
54,r03c05f01p01-ch1sk2fk1fl1.tiff,0,0.000581233,1e-06,1e-06,290.616,0.000
60,r03c05f02p01-ch1sk2fk1fl1.tiff,-0.000290616,0.001162465,1e-06,1e-06,0.000,581.232
66,r03c05f03p01-ch1sk2fk1fl1.tiff,0,0.001162465,1e-06,1e-06,290.616,581.232
72,r03c05f04p01-ch1sk2fk1fl1.tiff,0.000290616,0.001162465,1e-06,1e-06,581.232,581.232
78,r03c05f05p01-ch1sk2fk1fl1.tiff,0.000290616,0.000871849,1e-06,1e-06,581.232,290.616
84,r03c05f06p01-ch1sk2fk1fl1.tiff,0,0.000871849,1e-06,1e-06,290.616,290.616
90,r03c05f07p01-ch1sk2fk1fl1.tiff,-0.000290616,0.000871849,1e-06,1e-06,0.000,290.616
96,r03c05f08p01-ch1sk2fk1fl1.tiff,-0.000290616,0.000581233,1e-06,1e-06,0.000,0.000
102,r03c05f09p01-ch1sk2fk1fl1.tiff,0.000290616,0.000581233,1e-06,1e-06,581.232,0.000


## Compiled correclty but 8/9 positions are blank?

In [69]:
from pathlib import Path
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt

# Set your image folder path
image_dir = Path("data/untiled_images")

# Grab all .tif or .tiff files (non-recursive)
tif_files = sorted(image_dir.glob("*.tiff"))

# Load all images into a list
image_stack = []
for i, file in tqdm(enumerate(tif_files)):
    img = imread(file)
    image_stack.append(img)

# Convert to single 3D array: (N_images, H, W)
image_stack = np.stack(image_stack, axis=0)
print("Stacked image shape:", image_stack.shape)

# Find any blank (all-zero) images
blank_indices = [i for i, img in enumerate(image_stack) if np.all(img == 0)]
print(f"Blank images at indices: {blank_indices}")


0it [00:00, ?it/s]

Stacked image shape: (4050, 428, 428)
Blank images at indices: []


In [71]:
viewer.add_image(image_stack)
viewer.add_image(loaded_images)

<Image layer 'loaded_images' at 0x7fdb2aa53700>

INFO:OpenGL.acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'
Error calling Python override of QOpenGLWidget::event(): Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 655, in event
    out = super(QtBaseCanvasBackend, self).event(ev)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 655, in event
    out = super(QtBaseCanvasBackend, self).event(ev)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 963, in paintGL
    self._vispy_canvas.events.draw(region=None)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/home/dayn/miniconda3/envs/macrohet/lib/python3.10/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle